In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [3]:
import logging

import torch

from scripts import (
    generate_dataset,
    generate_pred_datasets,
    optuna_search,
    predictions,
    training,
)
from src.utils import configure_logger

logger = logging.getLogger(__name__)
configure_logger(logging.INFO, logging.INFO)

True

In [4]:
from qqe.src.experiments.plotting import plot_training_curves
from qqe.src.GNN.parameter_search.helpers import objective_GNN, objective_NN
from qqe.src.GNN.physics_aware_NN import GNN, NN, Regressor
from qqe.src.GNN.training.datasets import build_loaders, build_loaders_NN
from qqe.src.GNN.training.train import build_loss, train_model
from qqe.src.GNN.training.train_config import TrainConfig
from qqe.src.GNN.training.utils import (
    collect_dataset_paths,
    collect_files_path,
    evaluate_loss,
)
from qqe.src.GNN.training.runners import MODEL_REGISTRY

In [7]:
model_type="gnn"
epochs = 30
lr = 1e-4
loss_type = "huber"   # "mse" | "huber"
batch_size = 32
training_mode = "global"  # "global" | "per_family"
family = "clifford"  # required if training_mode == "per_family"
target = "sre"
data_dir = "../outputs/data/dataset"
model_save_path = "../outputs/models/clifford_model_gnn.pt"
show_progress=True
show_val_progress=False
log_every_n_batches=10
heartbeat_secs=60.0
epoch_time_warning_secs=600.0


model_hparams = {
    "gnn_hidden": 16,
    "gnn_heads": 6,
    "global_hidden": 128,
    "reg_hidden": 128,
    "num_layers": 3,
    "dropout_rate": 0.10,
}

train_hparams = {
    "weight_decay": 0.0003,
    "grad_clip": 0.04,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.0,
}

In [8]:
VALID_FAMILIES = {"haar", "clifford", "quansistor", "random"}
if model_type not in MODEL_REGISTRY:
    msg = f"Unsupported model_type: {model_type}. Must be one of {sorted(MODEL_REGISTRY)}"
    raise ValueError(msg)

if training_mode not in {"global", "per_family"}:
    msg = "training_mode must be 'global' or 'per_family'"
    raise ValueError(msg)

if training_mode == "per_family":
    if family is None:
        raise ValueError("family must be provided when training_mode='per_family'")
    if family not in VALID_FAMILIES:
        msg = f"Invalid family: {family}. Must be one of {sorted(VALID_FAMILIES)}"
        raise ValueError(
            msg,
        )

logger.info(f"Starting training | model_type={model_type} | training_mode={training_mode} | family={family} | loss_type={loss_type}")
cfg = TrainConfig(
    epochs=epochs,
    lr=lr,
    loss_type=loss_type,
    batch_size=batch_size,
    training_mode=training_mode,
    family=family,
    target=target,
    show_progress=show_progress,
    show_val_progress=show_val_progress,
    log_batch_loss_every=log_every_n_batches,
    heartbeat=heartbeat_secs,
    epoch_warning=epoch_time_warning_secs,
)
logger.info("Training configuration done.")

model_hparams = {} if model_hparams is None else dict(model_hparams)
train_hparams = {} if train_hparams is None else dict(train_hparams)

family_filter = family if training_mode == "per_family" else None
family_projection = family if training_mode == "per_family" else None

logger.info("Collecting data paths...")
# data_paths = collect_files_path(training_data_dir, family=family_filter)
train_paths = collect_dataset_paths(
    data_dir,
    family=family_filter,
    split="target",
)
if not train_paths:
    raise RuntimeError("No data paths found.")
logger.info(f"Found {len(train_paths)} data paths.")
logger.info("Data paths collected.")

2026-05-15 14:30:10,364 - __main__ - INFO - Starting training | model_type=gnn | training_mode=global | family=clifford | loss_type=huber
2026-05-15 14:30:10,364 - __main__ - INFO - Training configuration done.
2026-05-15 14:30:10,365 - __main__ - INFO - Collecting data paths...
2026-05-15 14:36:55,420 - __main__ - INFO - Found 107100 data paths.
2026-05-15 14:36:55,421 - __main__ - INFO - Data paths collected.


In [9]:
from collections.abc import Callable
from dataclasses import asdict
from pathlib import Path
from typing import Any

In [10]:
Loader = Callable[..., Any]

In [11]:
spec = MODEL_REGISTRY[model_type]
logger.info(f"Building loaders and model for model_type={model_type}...")

loader_fn: Loader = spec["build_loaders"]
returns_nodes_dim: bool = spec.get("returns_nodes_dim", False)
if returns_nodes_dim:
    train_loader, val_loader, test_loader, node_in_dim, global_in_dim, base_dataset = loader_fn(
        train_paths,
        batch_size=cfg.batch_size,
        seed=cfg.seed,
        train_split=cfg.train_split,
        val_split=cfg.val_split,
        global_feature_variant=cfg.global_feature_variant,
        node_feature_variant=cfg.node_feature_backend_variant,
        family_projection=family_projection,
    )
else:
    train_loader, val_loader, test_loader, global_in_dim, base_dataset = loader_fn(
        train_paths,
        batch_size=cfg.batch_size,
        seed=cfg.seed,
        train_split=cfg.train_split,
        val_split=cfg.val_split,
        global_feature_variant=cfg.global_feature_variant,
        node_feature_variant=cfg.node_feature_backend_variant,
        family_projection=family_projection,
    )
    node_in_dim = global_in_dim

model = spec["build_model"](node_in_dim, global_in_dim, model_hparams)
logger.info("Loaders and model built.")

2026-05-15 14:36:57,583 - __main__ - INFO - Building loaders and model for model_type=gnn...
2026-05-15 14:40:49,356 - __main__ - INFO - Loaders and model built.
